# EZhire Metrics Comparison
Loads saved models, computes scores, and writes ensemble artifacts for the Gradio app.

In [7]:
!pip install -q -U "sentence-transformers>=5.4.1,<6" "transformers>=4.41,<5" tf-keras datasets scikit-learn nltk PyMuPDF einops accelerate wordninja


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import os, re, json, warnings
from collections import Counter
from functools import lru_cache
import numpy as np
import pandas as pd
import nltk
import torch
import wordninja
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.preprocessing import normalize
from sklearn.linear_model import Ridge
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_absolute_error, ndcg_score, r2_score
from sklearn.model_selection import StratifiedKFold
from scipy.stats import spearmanr, pearsonr, rankdata
from nltk.corpus import stopwords, wordnet as wn
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from IPython.display import display

warnings.filterwarnings("ignore")
nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BASE_DIR = os.path.abspath(".")
MODEL_DIR = os.path.join(BASE_DIR, "saved_models")
ARTIFACT_DIR = os.path.join(BASE_DIR, "artifacts")
os.makedirs(ARTIFACT_DIR, exist_ok=True)

STOP_WORDS = set(stopwords.words("english"))
print(f"Device: {DEVICE}")

Device: cuda


In [9]:
ATS_MIN, ATS_MAX = 18.3, 90.7

def split_sep(text):
    if not isinstance(text, str): text = str(text)
    m = re.search(r"\[SEP\]|\bSEP\b", text)
    if m:
        a = text[:m.start()]
        b = text[m.end():]
        return a.strip(), b.strip()
    mid = len(text) // 2
    return text[:mid].strip(), text[mid:].strip()

def raw_text(text):
    if not isinstance(text, str): text = str(text)
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    lines = [re.sub(r'[ \t\f\v]+', ' ', l).strip() for l in text.split('\n')]
    return ' '.join(l for l in lines if l)

def clean_text(text):
    if not isinstance(text, str): text = str(text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text.lower())
    tokens = word_tokenize(re.sub(r'\s+', ' ', text).strip())
    return ' '.join(t for t in tokens if t not in STOP_WORDS and len(t) > 1)

def normalize_score(series):
    v = pd.to_numeric(series, errors='coerce').astype(float)
    return ((v - ATS_MIN) / (ATS_MAX - ATS_MIN)).clip(0, 1)

def denormalize_score(v):
    return np.asarray(v, float) * (ATS_MAX - ATS_MIN) + ATS_MIN

def build_df(src):
    splits = src['text'].apply(split_sep)
    ats = pd.to_numeric(src['ats_score'], errors='coerce').fillna(ATS_MIN)
    return pd.DataFrame({
        'resume_raw': splits.apply(lambda x: raw_text(x[0])),
        'jd_raw': splits.apply(lambda x: raw_text(x[1])),
        'resume_clean': splits.apply(lambda x: clean_text(x[0])),
        'jd_clean': splits.apply(lambda x: clean_text(x[1])),
        'original_label': src['original_label'].values,
        'ats_score_raw': ats.values,
        'ground_truth': normalize_score(ats).values
    }).dropna(subset=['resume_raw', 'jd_raw']).reset_index(drop=True)

ds = load_dataset('0xnbk/resume-ats-score-v1-en')
df_train = ds['train'].to_pandas()
df_val = ds['validation'].to_pandas()

df_tr = build_df(df_train)
df_vl = build_df(df_val)
print(f'Train: {len(df_tr)} | Val: {len(df_vl)}')
print(f'GT range train: {df_tr["ground_truth"].min():.3f} - {df_tr["ground_truth"].max():.3f}')

Train: 5099 | Val: 1275
GT range train: 0.012 - 0.991


## Shared Chunking Infrastructure

In [10]:
CHUNK_OVERLAP = 38
MAX_RESUME_CHUNKS = 10
MAX_JD_CHUNKS = 8
TOP_K_CHUNK_PAIRS = 5
ENCODE_BATCH = 16 if DEVICE == 'cuda' else 8

def extract_first_sentence(text, max_chars=150):
    t = raw_text(text)
    m = re.search(r'(?<=[a-zA-Z0-9])[.!?]', t)
    if m and m.start() > 10:
        sentence = t[:m.start() + 1].strip()
    else:
        sentence = t[:max_chars].strip()
    return sentence[:max_chars]

def token_chunk_body(body, tokenizer, max_tokens=512, overlap=CHUNK_OVERLAP, prefix_tokens=0):
    body = raw_text(body)
    if not body:
        return []
    ids = tokenizer.encode(body, add_special_tokens=False, truncation=False)
    effective_max = max(64, max_tokens - prefix_tokens - 2)
    if len(ids) <= effective_max:
        return [body]
    overlap = min(overlap, effective_max // 2)
    step = effective_max - overlap
    chunks = []
    for start in range(0, len(ids), step):
        piece = ids[start:start + effective_max]
        chunk = raw_text(tokenizer.decode(piece, skip_special_tokens=True))
        if chunk:
            chunks.append(chunk)
        if start + effective_max >= len(ids):
            break
    return chunks

def make_text_chunks(text, tokenizer, model_max_tokens=512, overlap=CHUNK_OVERLAP,
                     max_chunks=MAX_RESUME_CHUNKS, source_label='DOCUMENT'):
    text = raw_text(text)
    if not text:
        return []
    doc_ctx = extract_first_sentence(text)
    prefix = f'[DOC]: {doc_ctx} [{source_label}] ' if doc_ctx else f'[{source_label}] '
    prefix_tokens = len(tokenizer.encode(prefix, add_special_tokens=False))
    body_chunks = token_chunk_body(
        text, tokenizer,
        max_tokens=model_max_tokens,
        overlap=overlap,
        prefix_tokens=prefix_tokens
    )
    chunks = [f'{prefix}{c}' for c in body_chunks]
    if len(chunks) > max_chunks:
        keep = np.linspace(0, len(chunks) - 1, max_chunks, dtype=int).tolist()
        chunks = [chunks[i] for i in keep]
    fallback = text[:2000]
    return chunks or [f'{prefix}{fallback}']

def aggregate_chunk_sims(sims, top_k=TOP_K_CHUNK_PAIRS):
    sims = np.asarray(sims, float)
    if sims.size == 0: return 0.0
    flat = sims.reshape(-1)
    k = min(top_k, len(flat))
    top_mean = float(np.partition(flat, -k)[-k:].mean())
    coverage = float((sims.max(axis=0).mean() + sims.max(axis=1).mean()) / 2)
    return float(np.clip(0.75 * top_mean + 0.25 * coverage, 0.0, 1.0))

def score_doc_pair(model, t1, t2, left='RESUME', right='JOB'):
    tok = model.tokenizer
    mlen = model.max_seq_length
    c1 = make_text_chunks(t1, tok, model_max_tokens=mlen,
                          max_chunks=MAX_RESUME_CHUNKS, source_label=left)
    c2 = make_text_chunks(t2, tok, model_max_tokens=mlen,
                          max_chunks=MAX_JD_CHUNKS, source_label=right)
    e1 = model.encode(c1, convert_to_tensor=True, normalize_embeddings=True,
                      batch_size=ENCODE_BATCH, show_progress_bar=False)
    e2 = model.encode(c2, convert_to_tensor=True, normalize_embeddings=True,
                      batch_size=ENCODE_BATCH, show_progress_bar=False)
    sims = util.cos_sim(e1, e2).detach().cpu().numpy()
    return aggregate_chunk_sims(sims)

## Load Saved Models

In [11]:
def require_model(path, label):
    if not os.path.exists(os.path.join(path, 'modules.json')):
        raise FileNotFoundError(f'Missing {label} at {path}. Run the training notebook first.')

MPNET_PATH = os.path.join(MODEL_DIR, 'ezhire-mpnet_files')
ROBERTA_PATH = os.path.join(MODEL_DIR, 'ezhire-roberta-base_files')
BERT_PATH = os.path.join(MODEL_DIR, 'ezhire-bert-base_files')

require_model(MPNET_PATH, 'mpnet')
require_model(ROBERTA_PATH, 'roberta')
require_model(BERT_PATH, 'bert')

mpnet_model = SentenceTransformer(MPNET_PATH, device=DEVICE)
mpnet_model.max_seq_length = 384

roberta_model = SentenceTransformer(ROBERTA_PATH, device=DEVICE)
roberta_model.max_seq_length = 384

bert_model = SentenceTransformer(BERT_PATH, device=DEVICE)
bert_model.max_seq_length = 384

print('Models loaded successfully')

Models loaded successfully


In [12]:
print('Evaluating Model A (mpnet) on full validation set...')
mpnet_preds = []
for i, row in df_vl.iterrows():
    mpnet_preds.append(score_doc_pair(mpnet_model, row['resume_raw'], row['jd_raw']))
    if (i + 1) % 50 == 0: print(f'  {i + 1}/{len(df_vl)}')
df_vl['mpnet_score'] = [round(s * 100, 2) for s in mpnet_preds]
yt = df_vl['ground_truth'].values
yp = df_vl['mpnet_score'].values
mpnet_pearson = float(pearsonr(yp / 100, yt)[0])
mpnet_spearman = float(spearmanr(yp / 100, yt).correlation)
mpnet_mae = float(mean_absolute_error(yt, yp / 100))

print('Evaluating Model B (roberta) on full validation set...')
roberta_preds = []
for i, row in df_vl.iterrows():
    roberta_preds.append(score_doc_pair(roberta_model, row['resume_raw'], row['jd_raw']))
    if (i + 1) % 50 == 0: print(f'  {i + 1}/{len(df_vl)}')
df_vl['roberta_score'] = [round(s * 100, 2) for s in roberta_preds]
yp = df_vl['roberta_score'].values
roberta_pearson = float(pearsonr(yp / 100, yt)[0])
roberta_spearman = float(spearmanr(yp / 100, yt).correlation)
roberta_mae = float(mean_absolute_error(yt, yp / 100))

print('Evaluating Model C (bert) on full validation set...')
bert_preds = []
for i, row in df_vl.iterrows():
    bert_preds.append(score_doc_pair(bert_model, row['resume_raw'], row['jd_raw']))
    if (i + 1) % 50 == 0: print(f'  {i + 1}/{len(df_vl)}')
df_vl['bert_score'] = [round(s * 100, 2) for s in bert_preds]
yp = df_vl['bert_score'].values
bert_pearson = float(pearsonr(yp / 100, yt)[0])
bert_spearman = float(spearmanr(yp / 100, yt).correlation)
bert_mae = float(mean_absolute_error(yt, yp / 100))

print('Model A (mpnet) Results:')
print(f'  Pearson  : {mpnet_pearson:+.4f} | Spearman: {mpnet_spearman:+.4f} | MAE: {mpnet_mae:.4f}')
print('Model B (roberta) Results:')
print(f'  Pearson  : {roberta_pearson:+.4f} | Spearman: {roberta_spearman:+.4f} | MAE: {roberta_mae:.4f}')
print('Model C (bert) Results:')
print(f'  Pearson  : {bert_pearson:+.4f} | Spearman: {bert_spearman:+.4f} | MAE: {bert_mae:.4f}')

Token indices sequence length is longer than the specified maximum sequence length for this model (2739 > 384). Running this sequence through the model will result in indexing errors


Evaluating Model A (mpnet) on full validation set...
  50/1275
  100/1275
  150/1275
  200/1275
  250/1275
  300/1275
  350/1275
  400/1275
  450/1275
  500/1275
  550/1275
  600/1275
  650/1275
  700/1275
  750/1275
  800/1275
  850/1275
  900/1275
  950/1275
  1000/1275
  1050/1275
  1100/1275
  1150/1275
  1200/1275
  1250/1275


Token indices sequence length is longer than the specified maximum sequence length for this model (2695 > 384). Running this sequence through the model will result in indexing errors


Evaluating Model B (roberta) on full validation set...
  50/1275
  100/1275
  150/1275
  200/1275
  250/1275
  300/1275
  350/1275
  400/1275
  450/1275
  500/1275
  550/1275
  600/1275
  650/1275
  700/1275
  750/1275
  800/1275
  850/1275
  900/1275
  950/1275
  1000/1275
  1050/1275
  1100/1275
  1150/1275
  1200/1275
  1250/1275


Token indices sequence length is longer than the specified maximum sequence length for this model (2739 > 384). Running this sequence through the model will result in indexing errors


Evaluating Model C (bert) on full validation set...
  50/1275
  100/1275
  150/1275
  200/1275
  250/1275
  300/1275
  350/1275
  400/1275
  450/1275
  500/1275
  550/1275
  600/1275
  650/1275
  700/1275
  750/1275
  800/1275
  850/1275
  900/1275
  950/1275
  1000/1275
  1050/1275
  1100/1275
  1150/1275
  1200/1275
  1250/1275
Model A (mpnet) Results:
  Pearson  : +0.8447 | Spearman: +0.7673 | MAE: 0.1338
Model B (roberta) Results:
  Pearson  : +0.8422 | Spearman: +0.7674 | MAE: 0.1341
Model C (bert) Results:
  Pearson  : +0.8469 | Spearman: +0.7716 | MAE: 0.1330


In [13]:
print('Comparing Model A vs Model B vs Model C on ranking quality (NDCG, P@50, Spearman)...')

# Pick the SBERT arm for the ensemble by how well it RANKS resume/JD pairs. The
# downstream task is surfacing the best candidates (a ranking problem), so we select on
# ranking-oriented metrics rather than raw value-fidelity (Pearson). We deliberately use
# metrics that consume the WHOLE validation set so the choice is statistically stable:
#   - NDCG     : position-weighted quality of the full ranking (all 1275 pairs).
#   - P@50     : precision over the top 50 pooled pairs. (We use 50, not 5: the old P@5
#                / MRR rested on just 5 / 1 data points, swung in coarse 0.20 / 0.5 steps,
#                and manufactured a false "decisive" winner out of small-sample noise.)
#   - Spearman : rank correlation across all pairs (rank-based, so task-aligned).
# These helpers are also reused by full_metrics below.
def precision_at_k(yt, yp, k=5, thresh=0.5):
    top = sorted(zip(yp, yt), reverse=True)[:k]
    return sum(1 for _, t in top if t >= thresh) / k

def mrr_score(yt, yp, thresh=0.5):
    for rank, (_, t) in enumerate(sorted(zip(yp, yt), reverse=True), 1):
        if t >= thresh: return 1.0 / rank
    return 0.0

yt = df_vl['ground_truth'].values
sbert_score_cols = {'mpnet': 'mpnet_score', 'roberta': 'roberta_score', 'bert': 'bert_score'}

# combined = unweighted mean of the three ranking metrics (all in [0, 1]).
sbert_candidates = {}
print(f"  {'model':8s} {'NDCG':>7s} {'P@50':>6s} {'Spear':>7s} {'mean':>7s}")
for name, col in sbert_score_cols.items():
    yp = df_vl[col].values
    ypr = np.clip(yp / 100, 0, 1)
    nd = float(ndcg_score([yt], [yp]))
    p50 = precision_at_k(yt, ypr, 50)
    sp = float(spearmanr(ypr, yt).correlation)
    combined = (nd + p50 + sp) / 3
    sbert_candidates[name] = (combined, col, dict(ndcg=nd, p50=p50, spearman=sp))
    print(f'  {name:8s} {nd:7.4f} {p50:6.2f} {sp:7.4f} {combined:7.4f}')

best_sbert_name = max(sbert_candidates, key=lambda n: sbert_candidates[n][0])
best_sbert_score_col = sbert_candidates[best_sbert_name][1]
_label = {'mpnet': 'Model A (mpnet)', 'roberta': 'Model B (roberta)', 'bert': 'Model C (bert)'}
_m = sbert_candidates[best_sbert_name][2]
print(f'Best SBERT -> {_label[best_sbert_name]} | '
      f"NDCG={_m['ndcg']:.4f} P@50={_m['p50']:.2f} Spearman={_m['spearman']:.4f} "
      f'| ranking mean={sbert_candidates[best_sbert_name][0]:.4f}')
print('This model will be used as the SBERT component in the final ensemble.')

Comparing Model A vs Model B vs Model C on ranking quality (NDCG, P@50, Spearman)...
  model       NDCG   P@50   Spear    mean
  mpnet     0.9747   0.94  0.7673  0.8940
  roberta   0.9767   0.92  0.7674  0.8880
  bert      0.9725   0.94  0.7716  0.8947
Best SBERT -> Model C (bert) | NDCG=0.9725 P@50=0.94 Spearman=0.7716 | ranking mean=0.8947
This model will be used as the SBERT component in the final ensemble.


## Shared Segmentation + Lexical Preprocessing
Repairs fused tokens (`VistaWindows` → `vista windows`) so the lexical arms can match real words. Produces a desegmented `*_seg` text (TF-IDF) and lemmatized keyword lists (WordNet). Shared by both arms below.

In [14]:
# --- Targeted word-desegmentation -------------------------------------------
# The dataset text was extracted from PDFs and frequently drops spaces at
# formatting boundaries ("SummaryI", "TechnologyHeld", "98Windows"), which breaks
# the exact-token matching both lexical arms rely on. Repair the common cases
# cheaply (camelCase + letter/digit boundaries) and fall back to wordninja only for
# long all-lowercase blobs, so short tokens and acronyms (JDE, PMP) stay intact.
_CAMEL = re.compile(r'(?<=[a-z])(?=[A-Z])')
_LET_DIG = re.compile(r'(?<=[A-Za-z])(?=\d)|(?<=\d)(?=[A-Za-z])')

def desegment(text):
    if not isinstance(text, str): text = str(text)
    text = _CAMEL.sub(' ', text)
    text = _LET_DIG.sub(' ', text)
    out = []
    for tok in text.split():
        if tok.isalpha() and tok.islower() and len(tok) > 14:   # likely a fused blob
            out.extend(wordninja.split(tok))
        else:
            out.append(tok)
    return ' '.join(out)

# Representation for the TF-IDF arm: desegmented + lowercased, punctuation kept
# (the char n-gram analyzer uses it to match "c#", ".net", etc.).
def light_clean(text):
    t = desegment(raw_text(text)).lower()
    return re.sub(r'\s+', ' ', t).strip()

# Representation for the WordNet arm: desegmented, alpha-only, lemmatized; keep the
# k most frequent content lemmas to bound the pairwise synset comparisons.
_LEMM = WordNetLemmatizer()
def wn_tokens(text, k=40):
    t = re.sub(r'[^a-z\s]', ' ', desegment(raw_text(text)).lower())
    lemmas = [_LEMM.lemmatize(w) for w in t.split()
              if w not in STOP_WORDS and len(w) > 2]
    return [w for w, _ in Counter(lemmas).most_common(k)]

print('Building segmented / lemmatized text columns...')
for _df in (df_tr, df_vl):
    _df['resume_seg'] = _df['resume_raw'].apply(light_clean)
    _df['jd_seg'] = _df['jd_raw'].apply(light_clean)
df_vl['resume_kw'] = df_vl['resume_raw'].apply(lambda t: wn_tokens(t, 40))
df_vl['jd_kw'] = df_vl['jd_raw'].apply(lambda t: wn_tokens(t, 30))
print('Done. Example desegment:')
print('  ', desegment('SummaryI have 17 yearsExperience in VistaWindows98'))

Building segmented / lemmatized text columns...
Done. Example desegment:
   Summary I have 17 years Experience in Vista Windows 98


## TF-IDF Arm — corpus-fit, word + char
Fits a single vectorizer on the whole training corpus (real IDF), unioning word (1–2) and `char_wb` (3–5) features, then scores each validation pair by cosine on the segmented text.

In [15]:
# Corpus-fit TF-IDF: fit ONCE on all training resumes+JDs so IDF is meaningful
# (the old per-pair fit_transform([t1, t2]) gave a near-flat, useless IDF). Union a
# word arm (1-2 grams) with a char_wb arm (3-5) - the char arm matches substrings
# inside any residual fused tokens and survives tech punctuation like "c#"/".net".
word_vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True,
                           min_df=3, max_df=0.85, max_features=50000,
                           stop_words='english')
char_vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5),
                           sublinear_tf=True, min_df=3, max_features=50000)
tfidf_union = FeatureUnion([('word', word_vec), ('char', char_vec)])

corpus = pd.concat([df_tr['resume_seg'], df_tr['jd_seg']]).tolist()
print(f'Fitting TF-IDF union on {len(corpus)} train documents...')
tfidf_union.fit(corpus)

def rowwise_cosine(A, B):
    A, B = normalize(A), normalize(B)                  # L2-normalize rows
    return np.asarray(A.multiply(B).sum(axis=1)).ravel()

R = tfidf_union.transform(df_vl['resume_seg'])
J = tfidf_union.transform(df_vl['jd_seg'])
df_vl['tfidf_score'] = np.round(rowwise_cosine(R, J) * 100, 2)

_sp = spearmanr(df_vl['tfidf_score'].values, df_vl['ground_truth'].values).correlation
print(f'TF-IDF arm ready | feature dim = {R.shape[1]} | Spearman vs GT = {_sp:+.4f}')

Fitting TF-IDF union on 10198 train documents...
TF-IDF arm ready | feature dim = 100000 | Spearman vs GT = +0.1612


## WordNet Arm — lexical-semantic similarity
Synonym-aware soft overlap (Wu-Palmer) between resume and JD keywords. Decorrelated from both exact TF-IDF and neural SBERT, so it can add complementary signal to the ensemble.

In [16]:
# WordNet lexical-semantic overlap: bridges synonymy that exact TF-IDF misses.
# For each JD keyword, take the best Wu-Palmer similarity to any resume keyword,
# then average over JD keywords (a soft, synonym-aware coverage score in [0,1]).
@lru_cache(maxsize=200000)
def _first_synset(word):
    s = wn.synsets(word)
    return s[0].name() if s else None

@lru_cache(maxsize=2000000)
def _wup(name_a, name_b):
    v = wn.synset(name_a).wup_similarity(wn.synset(name_b))
    return float(v) if v else 0.0

def wordnet_pair_score(resume_kw, jd_kw):
    if not jd_kw:
        return 0.0
    r_syn = [(w, _first_synset(w)) for w in resume_kw]
    r_set = {w for w, _ in r_syn}
    total = 0.0
    for jw in jd_kw:
        if jw in r_set:                      # exact lemma match
            total += 1.0
            continue
        js = _first_synset(jw)
        best = 0.0
        if js is not None:
            for rw, rs in r_syn:
                if rs is not None:
                    v = _wup(js, rs)
                    if v > best:
                        best = v
                        if best >= 0.999:
                            break
        total += best
    return total / len(jd_kw)

print('Computing WordNet arm scores...')
wn_preds = []
for i, row in df_vl.iterrows():
    wn_preds.append(wordnet_pair_score(row['resume_kw'], row['jd_kw']))
    if (i + 1) % 200 == 0:
        print(f'  {i + 1}/{len(df_vl)}')
df_vl['wordnet_score'] = np.round(np.array(wn_preds) * 100, 2)

_sp = spearmanr(df_vl['wordnet_score'].values, df_vl['ground_truth'].values).correlation
print(f'WordNet arm ready | Spearman vs GT = {_sp:+.4f}')

Computing WordNet arm scores...
  200/1275
  400/1275
  600/1275
  800/1275
  1000/1275
  1200/1275
WordNet arm ready | Spearman vs GT = +0.1269


In [17]:
# Put every arm on the SAME scale before blending. We optimize NDCG (a ranking
# metric), so rank-normalization is the natural choice: it removes the variance
# mismatch that previously forced TF-IDF to weight 0.
def rank_norm(x):
    x = np.asarray(x, float)
    return rankdata(x) / len(x)

best_sbert_preds = df_vl[best_sbert_score_col].values
yt = df_vl['ground_truth'].values
tiers = df_vl['original_label'].values

feat = {
    'sbert':   rank_norm(best_sbert_preds),
    'tfidf':   rank_norm(df_vl['tfidf_score'].values),
    'wordnet': rank_norm(df_vl['wordnet_score'].values),
}
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# --- (1) Sanity check: rank-normalized linear blend of SBERT + TF-IDF ---------
# With both arms rank-normalized, search the FULL weight range (the old grid only
# went 0.50-1.00). If TF-IDF now earns weight > 0, the scale fix worked. We score
# each fold by NDCG (ranking quality), the metric we optimize for.
weight_results = {}
for w in np.round(np.arange(0.0, 1.01, 0.05), 2):
    fold = []
    for _, va in kf.split(np.zeros(len(yt)), tiers):
        ens = w * feat['sbert'][va] + (1 - w) * feat['tfidf'][va]
        fold.append(ndcg_score([yt[va]], [ens]))
    weight_results[w] = round(float(np.mean(fold)), 4)
BEST_W = max(weight_results, key=weight_results.get)
print(f'Rank-normalized linear blend (SBERT+TF-IDF): best SBERT weight={BEST_W:.2f} '
      f'| TF-IDF weight={1 - BEST_W:.2f} | NDCG={weight_results[BEST_W]:.4f}')

# --- (2) Stacked ensemble: out-of-fold Ridge over rank-normalized arms ---------
# A meta-learner handles scale + complementary signal better than a single weight,
# and out-of-fold prediction keeps the comparison honest (no in-sample optimism).
# The Ridge still regresses on the ground-truth scores; we just SELECT the config
# by NDCG (ranking quality) rather than Spearman.
def oof_stack(cols, alpha=1.0):
    X = np.column_stack([feat[c] for c in cols])
    oof = np.zeros(len(yt))
    for tr, va in kf.split(X, tiers):
        m = Ridge(alpha=alpha).fit(X[tr], yt[tr])
        oof[va] = m.predict(X[va])
    return float(ndcg_score([yt], [oof])), oof

CONFIGS = {
    'SBERT only':               ['sbert'],
    'SBERT + TF-IDF':           ['sbert', 'tfidf'],
    'SBERT + WordNet':          ['sbert', 'wordnet'],
    'SBERT + TF-IDF + WordNet': ['sbert', 'tfidf', 'wordnet'],
}
print('\n4-way out-of-fold stacked comparison (Ridge over rank-normalized arms):')
oof_results = {}
for name, cols in CONFIGS.items():
    nd, oof = oof_stack(cols)
    oof_results[name] = (nd, oof)
    print(f'  {name:28s} OOF NDCG = {nd:.4f}')

BEST_CONFIG = max(oof_results, key=lambda k: oof_results[k][0])
best_oof = oof_results[BEST_CONFIG][1]
print(f'\nBest ensemble config: {BEST_CONFIG} '
      f'(NDCG={oof_results[BEST_CONFIG][0]:.4f})')

# Production scores: use the out-of-fold predictions (already 0-1 GT scale -> *100).
def get_tier(score):
    if score >= 70: return 'Strong Match'
    if score >= 45: return 'Potential Fit'
    return 'Poor Match'

df_vl['ensemble_score'] = np.round(np.clip(best_oof, 0, 1) * 100, 2)
df_vl['tier'] = df_vl['ensemble_score'].apply(get_tier)
df_sample = df_vl.copy()
print('Ensemble scoring complete.')

Rank-normalized linear blend (SBERT+TF-IDF): best SBERT weight=0.95 | TF-IDF weight=0.05 | NDCG=0.9697

4-way out-of-fold stacked comparison (Ridge over rank-normalized arms):
  SBERT only                   OOF NDCG = 0.9753
  SBERT + TF-IDF               OOF NDCG = 0.9740
  SBERT + WordNet              OOF NDCG = 0.9754
  SBERT + TF-IDF + WordNet     OOF NDCG = 0.9735

Best ensemble config: SBERT + WordNet (NDCG=0.9754)
Ensemble scoring complete.


In [18]:
def safe_r(x, y, kind='pearson'):
    if len(x) < 2 or np.std(x) == 0 or np.std(y) == 0: return 0.0
    v = pearsonr(x, y)[0] if kind == 'pearson' else spearmanr(x, y).correlation
    return 0.0 if np.isnan(v) else float(v)

def precision_at_k(yt, yp, k=5, thresh=0.5):
    top = sorted(zip(yp, yt), reverse=True)[:k]
    return sum(1 for _, t in top if t >= thresh) / k

def mrr_score(yt, yp, thresh=0.5):
    for rank, (_, t) in enumerate(sorted(zip(yp, yt), reverse=True), 1):
        if t >= thresh: return 1.0 / rank
    return 0.0

def full_metrics(yt_norm, yp_pct, name):
    yt = np.asarray(yt_norm, float)
    yp = np.asarray(yp_pct, float)
    ypr = np.clip(yp / 100, 0, 1)
    mask = ~np.isnan(yt) & ~np.isnan(yp)
    yt, yp, ypr = yt[mask], yp[mask], ypr[mask]
    if len(yt) < 2: return {}
    # We report the three metrics we actually select on - NDCG, P@50, Spearman - all
    # computed over the FULL validation set, so they're statistically stable. P@5/P@10/
    # MRR are still computed (and kept in the saved table for reference / the dashboard)
    # but they rest on only 5 / 1 pooled points and swing in coarse steps, so they're
    # not shown here. Pearson/MAE measure value-fidelity, not ranking.
    nd = ndcg_score([yt], [yp])
    p50 = precision_at_k(yt, ypr, 50)
    sp = safe_r(ypr, yt, 'spearman')
    p5 = precision_at_k(yt, ypr, 5)
    p10 = precision_at_k(yt, ypr, 10)
    m = mrr_score(yt, ypr)
    pe = safe_r(ypr, yt, 'pearson')
    mae = mean_absolute_error(yt, ypr)
    print(f'{name:28s} NDCG={nd:.4f} P@50={p50:.3f} Spearman={sp:+.4f}')
    return dict(model=name, ndcg=nd, precision_at_50=p50, spearman=sp,
                precision_at_5=p5, precision_at_10=p10, mrr=m, pearson=pe, mae=mae)

print('Full metrics on validation set (showing NDCG, P@50, Spearman):')
yt = df_sample['ground_truth'].values
r_mpnet = full_metrics(yt, df_sample['mpnet_score'].values, 'mpnet')
r_roberta = full_metrics(yt, df_sample['roberta_score'].values, 'roberta')
r_bert = full_metrics(yt, df_sample['bert_score'].values, 'bert')
r_tfidf = full_metrics(yt, df_sample['tfidf_score'].values, 'TF-IDF')
r_wordnet = full_metrics(yt, df_sample['wordnet_score'].values, 'WordNet')
r_ensemble = full_metrics(yt, df_sample['ensemble_score'].values, f'Ensemble[{BEST_CONFIG}]')

metrics_df = pd.DataFrame([r_mpnet, r_roberta, r_bert, r_tfidf, r_wordnet, r_ensemble])
if 'model' in metrics_df.columns:
    metrics_df = metrics_df.set_index('model')
print('Summary Table (NDCG, P@50, Spearman):')
display(metrics_df[['ndcg', 'precision_at_50', 'spearman']].round(4))

Full metrics on validation set (showing NDCG, P@50, Spearman):
mpnet                        NDCG=0.9747 P@50=0.940 Spearman=+0.7673
roberta                      NDCG=0.9767 P@50=0.920 Spearman=+0.7674
bert                         NDCG=0.9725 P@50=0.940 Spearman=+0.7716
TF-IDF                       NDCG=0.8796 P@50=0.580 Spearman=+0.1612
WordNet                      NDCG=0.8853 P@50=0.600 Spearman=+0.1269
Ensemble[SBERT + WordNet]    NDCG=0.9754 P@50=0.900 Spearman=+0.7729
Summary Table (NDCG, P@50, Spearman):


,ndcg,precision_at_50,spearman
model,,,
mpnet,0.9747,0.94,0.7673
roberta,0.9767,0.92,0.7674
bert,0.9725,0.94,0.7716
TF-IDF,0.8796,0.58,0.1612
WordNet,0.8853,0.60,0.1269
Ensemble[SBERT + WordNet],0.9754,0.90,0.7729


In [19]:
import joblib

metrics_path = os.path.join(ARTIFACT_DIR, 'metrics_df.csv')
scores_path = os.path.join(ARTIFACT_DIR, 'validation_scores.csv')
config_path = os.path.join(ARTIFACT_DIR, 'ensemble_config.json')
vectorizer_path = os.path.join(ARTIFACT_DIR, 'tfidf_union.joblib')
ensemble_model_path = os.path.join(ARTIFACT_DIR, 'ensemble_model.joblib')

metrics_df.reset_index().to_csv(metrics_path, index=False)
df_sample[[
    'resume_raw', 'jd_raw', 'original_label', 'ats_score_raw', 'ground_truth',
    'mpnet_score', 'roberta_score', 'bert_score', 'tfidf_score', 'wordnet_score',
    'ensemble_score', 'tier'
]].to_csv(scores_path, index=False)

with open(config_path, 'w', encoding='utf-8') as f:
    json.dump({
        'best_sbert_name': best_sbert_name,
        'best_sbert_weight': float(BEST_W),            # rank-normalized SBERT+TF-IDF linear blend
        'best_ensemble_config': BEST_CONFIG,
        'ensemble_arms': CONFIGS[BEST_CONFIG],
        'oof_ndcg': {k: round(v[0], 4) for k, v in oof_results.items()}
    }, f, indent=2)

# Persist the corpus-fit TF-IDF vectorizer so the Gradio app can reuse the trained
# IDF for keyword extraction / lexical similarity instead of fitting per-document.
joblib.dump(tfidf_union, vectorizer_path)

# Persist the PRODUCTION ensemble so the Gradio app scores new pairs with the same
# winning combination (Ridge over rank-normalized arms), not just the SBERT arm.
# We refit one Ridge on all validation rows and save the per-arm score distributions
# so the app can rank-normalize a new pair's arm scores exactly as the Ridge expects.
ensemble_arms = CONFIGS[BEST_CONFIG]
arm_col = {'sbert': best_sbert_score_col, 'tfidf': 'tfidf_score', 'wordnet': 'wordnet_score'}
final_ridge = Ridge(alpha=1.0).fit(np.column_stack([feat[a] for a in ensemble_arms]), yt)
ensemble_refs = {a: np.sort(df_vl[arm_col[a]].values.astype(float)) for a in ensemble_arms}
joblib.dump({
    'ridge': final_ridge,
    'arms': list(ensemble_arms),
    'refs': ensemble_refs,
    'sbert_name': best_sbert_name,
}, ensemble_model_path)

print(f'Wrote metrics to: {metrics_path}')
print(f'Wrote validation scores to: {scores_path}')
print(f'Wrote ensemble config to: {config_path}')
print(f'Wrote TF-IDF vectorizer to: {vectorizer_path}')
print(f'Wrote ensemble model to: {ensemble_model_path}')
print('Ensemble coefficients:',
      {a: round(float(c), 4) for a, c in zip(ensemble_arms, final_ridge.coef_)},
      '| intercept', round(float(final_ridge.intercept_), 4))

Wrote metrics to: c:\Users\Syakir\Downloads\Projects\nlp final project 2\EZhire_final_pipeline\artifacts\metrics_df.csv
Wrote validation scores to: c:\Users\Syakir\Downloads\Projects\nlp final project 2\EZhire_final_pipeline\artifacts\validation_scores.csv
Wrote ensemble config to: c:\Users\Syakir\Downloads\Projects\nlp final project 2\EZhire_final_pipeline\artifacts\ensemble_config.json
Wrote TF-IDF vectorizer to: c:\Users\Syakir\Downloads\Projects\nlp final project 2\EZhire_final_pipeline\artifacts\tfidf_union.joblib
Wrote ensemble model to: c:\Users\Syakir\Downloads\Projects\nlp final project 2\EZhire_final_pipeline\artifacts\ensemble_model.joblib
Ensemble coefficients: {'sbert': 0.9931, 'wordnet': -0.0703} | intercept -0.0595
